# 20.2 Data pipelines & feature stores

Feature stores are contracts: the same value must be computed, timestamped, joined, and served the same way offline and online.

Data splitting and leakage control feed the point-in-time join; feature engineering feeds the reusable transformation contract.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(202)


def summarize_feature_workload(workload):
    row_count = int(workload["event_time"].size)
    ages = workload["read_time"] - workload["event_time"]
    violation_rate = float(np.mean((ages > workload["sla_minutes"]) | workload["missing"]))
    coverage = float(np.mean(~workload["missing"]))
    throughput = row_count / max(float(workload["duration_minutes"] * 60.0), 1.0)
    return {
        "name": workload["name"],
        "requests": row_count,
        "p50_ms": float(np.percentile(workload["latency_ms"], 50)),
        "p95_ms": float(np.percentile(workload["latency_ms"], 95)),
        "throughput_qps": throughput,
        "cost_dollars": float(np.sum(workload["cost_usd"])),
        "drift_stat": float(abs(np.mean(workload["online_value"]) - np.mean(workload["offline_value"]))),
        "violation_rate": violation_rate,
        "coverage": coverage,
    }


def simulate_feature_workload(name, row_count, late_rate=0.0, missing_rate=0.0, skew=0.0, hourly=False):
    event_time = RNG.uniform(0.0, 240.0, row_count)
    delay = RNG.lognormal(mean=np.log(12.0), sigma=0.45, size=row_count)
    if late_rate > 0.0:
        late_mask = RNG.random(row_count) < late_rate
        delay[late_mask] += RNG.uniform(35.0, 120.0, int(np.sum(late_mask)))
    read_time = event_time + delay
    offline_value = RNG.normal(0.42, 0.08, row_count)
    online_value = offline_value + RNG.normal(skew, 0.03, row_count)
    missing = RNG.random(row_count) < missing_rate
    latency_ms = RNG.lognormal(mean=np.log(18.0), sigma=0.35, size=row_count)
    cost_usd = 0.000003 * np.ones(row_count)
    duration_minutes = 60.0 if hourly else max(float(np.max(read_time) - np.min(event_time)), 1.0)
    return {
        "name": name,
        "event_time": event_time,
        "read_time": read_time,
        "label_time": event_time + RNG.uniform(10.0, 40.0, row_count),
        "feature_time": event_time + delay,
        "offline_value": offline_value,
        "online_value": online_value,
        "missing": missing,
        "latency_ms": latency_ms,
        "cost_usd": cost_usd,
        "duration_minutes": duration_minutes,
        "sla_minutes": 30.0,
    }


def make_workload_ladder():
    d1 = {
        "name": "D1 tiny feature/event/label table",
        "event_time": np.array([100.0, 104.0, 111.0]),
        "read_time": np.array([118.0, 125.0, 146.0]),
        "label_time": np.array([120.0, 130.0, 150.0]),
        "feature_time": np.array([118.0, 125.0, 146.0]),
        "offline_value": np.array([0.40, 0.43, 0.44]),
        "online_value": np.array([0.41, 0.44, 0.45]),
        "missing": np.array([False, False, True]),
        "latency_ms": np.array([15.0, 18.0, 21.0]),
        "cost_usd": np.array([0.000003, 0.000003, 0.000003]),
        "duration_minutes": 1.0,
        "sla_minutes": 30.0,
    }
    d2 = simulate_feature_workload("D2 10k clean feature rows", 10_000)
    d3 = simulate_feature_workload("D3 late arrivals, missingness, skew", 40_000, late_rate=0.08, missing_rate=0.06, skew=0.05)
    d4 = simulate_feature_workload("D4 real-ish event-time trace", 180_000, late_rate=0.04, missing_rate=0.03, skew=0.025, hourly=True)
    d5 = simulate_feature_workload("D5 production-scale hourly feature store", 1_000_000, late_rate=0.06, missing_rate=0.04, skew=0.04, hourly=True)
    return [d1, d2, d3, d4, d5]


## The concept, built once: point-in-time joins

The lesson quantities are $\text{age}=t_{read}-t_{event}$ and $\text{coverage}=n_{present}/n_{total}$. We first implement a reusable point-in-time filter and assert $118-100=18$ plus the future-leakage gap $125-120=5$.

In [ ]:

def point_in_time_join(feature_time, label_time, read_time, present):
    age = read_time - feature_time
    valid_time = feature_time <= label_time
    usable = valid_time & present
    coverage = float(np.mean(usable))
    return age, usable, coverage

age_example = 118 - 100
leak_gap = 125 - 120
assert age_example == 18
assert leak_gap == 5
print("freshness age", age_example)
print("future leakage minutes", leak_gap)


On the D1 table, coverage is the fraction of usable values. The lesson's worked number $92/100=0.92$ is asserted separately so the implementation and lesson arithmetic stay aligned.

In [ ]:

feature_time = np.array([100.0, 114.0, 125.0])
label_time = np.array([120.0, 130.0, 120.0])
read_time = np.array([118.0, 128.0, 130.0])
present = np.array([True, True, True])
age, usable, d1_coverage = point_in_time_join(feature_time, label_time, read_time, present)
lesson_coverage = 92 / 100
assert np.isclose(lesson_coverage, 0.92)
assert usable.tolist() == [True, True, False]
print("D1 usable mask", usable)
print("lesson coverage", lesson_coverage)


## The dataset ladder

Family F17 has no shared ML-training ladder here. We build the operational workload ladder inline: D1 tiny trace, D2 larger volume, D3 spikes or drift, D4 real-ish synthetic trace, and D5 production-scale simulation.

In [ ]:

workloads = make_workload_ladder()
for workload in workloads:
    print(workload["name"])
    print("shape", workload["event_time"].shape)
    print("sample ages", np.round((workload["read_time"] - workload["event_time"])[:5], 2))


## Run the same method across D1-D5

The metric is freshness SLA violation rate. A row violates if its age exceeds the SLA or the feature is missing; the same summary also reports latency, cost, throughput, and online/offline drift.

In [ ]:

workloads = make_workload_ladder()
summaries = [summarize_feature_workload(workload) for workload in workloads]

header = f"{'rung':<42} {'load':>10} {'p50':>9} {'p95':>9} {'cost':>10} {'qps':>10} {'violation_rate':>12}"
print(header)
for row in summaries:
    line = f"{row['name']:<42} {row['requests']:>10} {row['p50_ms']:>9.2f} {row['p95_ms']:>9.2f} {row['cost_dollars']:>10.3f} {row['throughput_qps']:>10.2f} {row['violation_rate']:>12.5f}"
    print(line)


The first row is the operational artifact: p95 latency, total cost, and throughput by rung. The second plot is the lesson metric versus load.

In [ ]:

names = [row["name"].split()[0] for row in summaries]
loads = np.array([row["requests"] for row in summaries], dtype=float)
p95_values = np.array([row["p95_ms"] for row in summaries], dtype=float)
cost_values = np.array([row["cost_dollars"] for row in summaries], dtype=float)
throughput_values = np.array([row["throughput_qps"] for row in summaries], dtype=float)
metric_values = np.array([row["violation_rate"] for row in summaries], dtype=float)
cost_per_request = cost_values / loads
normalized_p95 = p95_values / max(float(np.max(p95_values)), 1.0)
normalized_cost = cost_per_request / max(float(np.max(cost_per_request)), 1.0)
normalized_throughput = throughput_values / max(float(np.max(throughput_values)), 1.0)

fig = plt.figure(figsize=(15, 7))
grid = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.15])
for index, name in enumerate(names):
    ax = fig.add_subplot(grid[0, index])
    values = [normalized_p95[index], normalized_cost[index], normalized_throughput[index]]
    ax.bar(["p95", "cost", "qps"], values)
    ax.set_ylim(0.0, 1.05)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("normalized")
summary_ax = fig.add_subplot(grid[1, :])
summary_ax.plot(loads, metric_values, marker="o")
summary_ax.set_xscale("log")
summary_ax.set_title("freshness violation rate vs load")
summary_ax.set_xlabel("records or requests")
summary_ax.set_ylabel("freshness violation rate")
fig.suptitle("Per-workload latency, cost, throughput, and metric-vs-load")
fig.tight_layout()


## Pitfall on D5: future leakage

A leaky join accepts feature timestamps after the label time and inflates coverage. The fix is the point-in-time mask $t_{feature}\le t_{label}$ before computing coverage or metrics.

In [ ]:

d5 = workloads[-1]
naive_coverage = float(np.mean(~d5["missing"]))
valid_time = d5["feature_time"] <= d5["label_time"]
fixed_coverage = float(np.mean((~d5["missing"]) & valid_time))
leaky_rows = int(np.sum((~d5["missing"]) & (~valid_time)))
print("naive coverage", round(naive_coverage, 4))
print("fixed point-in-time coverage", round(fixed_coverage, 4))
print("leaky rows removed", leaky_rows)


## Evaluate it + Practice

- Metric: freshness SLA violation rate; no-skill baseline is joining the latest feature regardless of label time.
- Sanity check: event-time age must be nonnegative for served features.
- Ablation: remove the point-in-time filter and watch coverage rise for the wrong reason.
- Failure signals: violation rate spikes, online/offline means separate, or missingness clusters by hour.

Practice: change the SLA from 30 to 20 minutes and recompute violations.

Practice: inject a larger online skew and inspect the drift statistic.

Practice: count how many D5 rows are excluded only because of future leakage.